In [9]:
import pandas as pd
import numpy as np
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import plotly.io as pio
import json

PROCESSED_DIR = Path("../data/processed")
OUTPUT_DIR = Path("../output")
CHART_DIR = OUTPUT_DIR / "charts"
EXPORT_DIR = OUTPUT_DIR / "final_exports"

CHART_DIR.mkdir(parents=True, exist_ok=True)
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

SCHOOLS_EXPOSURE_PATH = PROCESSED_DIR / "schools_with_exposure.csv"
DISTRICT_SUMMARY_PATH = PROCESSED_DIR / "district_exposure_summary.csv"
TYPE_SUMMARY_PATH = PROCESSED_DIR / "type_exposure_summary.csv"
LEVEL_SUMMARY_PATH = PROCESSED_DIR / "level_exposure_summary.csv"
TOP_EXPOSED_PATH = PROCESSED_DIR / "top_exposed_schools.csv"
FARTHEST_PATH = PROCESSED_DIR / "farthest_schools.csv"
STATION_COVERAGE_PATH = PROCESSED_DIR / "station_coverage_summary.csv"
CONFIDENCE_PATH = PROCESSED_DIR / "exposure_confidence_summary.csv"

##### Load Tables

In [10]:
schools = pd.read_csv(SCHOOLS_EXPOSURE_PATH)
district_summary = pd.read_csv(DISTRICT_SUMMARY_PATH)
type_summary = pd.read_csv(TYPE_SUMMARY_PATH)
level_summary = pd.read_csv(LEVEL_SUMMARY_PATH)
top_exposed = pd.read_csv(TOP_EXPOSED_PATH)
farthest = pd.read_csv(FARTHEST_PATH)
station_coverage = pd.read_csv(STATION_COVERAGE_PATH)
confidence_summary = pd.read_csv(CONFIDENCE_PATH)

print("schools:", schools.shape)
print("district_summary:", district_summary.shape)
print("type_summary:", type_summary.shape)
print("level_summary:", level_summary.shape)
print("top_exposed:", top_exposed.shape)
print("farthest:", farthest.shape)
print("station_coverage:", station_coverage.shape)
print("confidence_summary:", confidence_summary.shape)

schools: (2423, 61)
district_summary: (16, 9)
type_summary: (2, 9)
level_summary: (16, 7)
top_exposed: (2423, 11)
farthest: (2423, 9)
station_coverage: (44, 7)
confidence_summary: (3, 5)


##### Build KPI Table
This creates the small numbers that usually appear at the top of a dashboard.

In [11]:
kpi = pd.DataFrame([
    {"metric": "total_schools", "value": len(schools)},
    {"metric": "district_count", "value": schools["District"].nunique()},
    {"metric": "station_count_used", "value": schools["nearest_station_id"].nunique()},
    {"metric": "avg_pm25_all", "value": round(schools["PM2.5 (µg/m³)_mean"].mean(), 2)},
    {"metric": "median_pm25_all", "value": round(schools["PM2.5 (µg/m³)_mean"].median(), 2)},
    {"metric": "avg_distance_km", "value": round(schools["distance_km"].mean(), 2)},
    {"metric": "max_distance_km", "value": round(schools["distance_km"].max(), 2)},
    {"metric": "high_confidence_schools", "value": int((schools["exposure_confidence"] == "High").sum())},
    {"metric": "medium_confidence_schools", "value": int((schools["exposure_confidence"] == "Medium").sum())},
    {"metric": "low_confidence_schools", "value": int((schools["exposure_confidence"] == "Low").sum())},
])

kpi

,metric,value
0,total_schools,2423.00
1,district_count,16.00
2,station_count_used,44.00
3,avg_pm25_all,98.04
4,median_pm25_all,97.67
5,avg_distance_km,2.66
6,max_distance_km,15.20
7,high_confidence_schools,2260.00
8,medium_confidence_schools,154.00
9,low_confidence_schools,9.00


##### Build Dashboard-Ready Tables
These are trimmed, smaller tables meant for charts, ranking cards, and dashboard widgets.

In [12]:
chart_districts = district_summary.sort_values("avg_pm25", ascending=False).head(10).copy()
dashboard_type = type_summary.copy()
chart_top_exposed = top_exposed.head(20).copy()
chart_farthest = farthest.head(20).copy()
chart_station_coverage = station_coverage.sort_values("school_count", ascending=False).head(15).copy()
chart_conf = confidence_summary.copy()

print(chart_districts.shape)
print(dashboard_type.shape)
print(chart_top_exposed.shape)
print(chart_farthest.shape)
print(chart_station_coverage.shape)
print(chart_conf.shape)

(10, 9)
(2, 9)
(20, 11)
(20, 9)
(15, 7)
(3, 5)


#####  Export Dashboard CSVs

In [13]:
kpi.to_csv(EXPORT_DIR / "dashboard_kpi.csv", index=False)
chart_districts.to_csv(EXPORT_DIR / "dashboard_district_top10.csv", index=False)
dashboard_type.to_csv(EXPORT_DIR / "dashboard_type_summary.csv", index=False)
level_summary.to_csv(EXPORT_DIR / "dashboard_level_summary.csv", index=False)
chart_top_exposed.to_csv(EXPORT_DIR / "dashboard_top_exposed_top20.csv", index=False)
chart_farthest.to_csv(EXPORT_DIR / "dashboard_farthest_top20.csv", index=False)
chart_station_coverage.to_csv(EXPORT_DIR / "dashboard_station_coverage_top15.csv", index=False)
chart_conf.to_csv(EXPORT_DIR / "dashboard_confidence_summary.csv", index=False)

print("Dashboard CSVs exported")

Dashboard CSVs exported


In [14]:
def save_meta(path, caption, description):
    meta_path = str(path) + ".meta.json"
    with open(meta_path, "w", encoding="utf-8") as f:
        json.dump(
            {"caption": caption, "description": description},
            f,
            ensure_ascii=False,
            indent=2
        )

In [24]:
fig = px.bar(
    chart_districts,
    x="District",
    y="avg_pm25",
    title="District PM2.5 exposure (top 10)"
)

fig.update_layout(
    title={
        "text": "District PM2.5 exposure (top 10)<br><span style='font-size: 18px; font-weight: normal;'>Average school exposure by district | Higher means worse exposure</span>"
    }
)
fig.update_xaxes(title_text="District")
fig.update_yaxes(title_text="PM2.5 mean")
fig.update_traces(cliponaxis=False)

fig.write_image(CHART_DIR / "district_pm25_top10.png")
save_meta(
    CHART_DIR / "district_pm25_top10.png",
    "Top 10 districts by average PM2.5 exposure",
    "Bar chart of the ten districts with the highest average school PM2.5 exposure."
)

ValueError: 
Image export using the "kaleido" engine requires the Kaleido package,
which can be installed using pip:

    $ pip install --upgrade kaleido


In [17]:
import sys
print(sys.executable)

import plotly
print(plotly.__version__)

/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/bin/python
6.9.0


In [18]:
import sys
!{sys.executable} -m pip install -U kaleido

zsh:1: no such file or directory: /Users/kumarb/Documents/My


In [20]:
import sys
!python -m pip install -U kaleido

You should consider upgrading via the '/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/bin/python -m pip install --upgrade pip' command.


In [22]:
import sys
!"{sys.executable}" -m pip install -U kaleido

You should consider upgrading via the '/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/bin/python -m pip install --upgrade pip' command.


In [23]:
pip install -U kaleido

You should consider upgrading via the '/Users/kumarb/Documents/My Projects/delhi_school_air_pollution/.venv/bin/python -m pip install --upgrade pip' command.
Note: you may need to restart the kernel to use updated packages.
